# PanAf Ape Detection — Phase 1 "See"

**Pretrained MegaDetector V6 over PanAf500 clips, on a Colab GPU.**

This notebook runs the real pipeline: it downloads a purposive sample of PanAf500 clips, runs
detection over every frame, stitches annotated video, and measures accuracy against the dataset's
ground-truth boxes.

Roughly **10–20 minutes** for 10 clips on a T4.

### Before you start

**Runtime → Change runtime type → T4 GPU**, *then* run the cells. Changing the runtime restarts the
session and discards anything installed.

### Two things this notebook is careful about

1. **The device.** PyTorch-Wildlife accepts `device="cuda"`, stores it, and **never applies it** —
   the weights load on CPU and nothing raises. On Colab that is CPU speed with CUDA in the metadata.
   The pipeline forces and then *verifies* the device; section 4 shows you the check.
2. **Prediction vs ground truth.** MegaDetector emits `animal` — not species, not behaviour. In the
   annotated video, **green boxes are predictions** and **amber boxes are dataset ground truth**.
   Behaviour labels come from the dataset, never from the model.

### Licence

PanAf20K is released under a **Non-Commercial Government Licence v2**. Clips downloaded here must
not be redistributed, and the annotated video you produce is a derived work.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi || echo "NO GPU — Runtime > Change runtime type > T4 GPU, then Runtime > Restart session."

## 2. Clone the repository

Replace `REPO_URL` if you are working from a fork.

In [ ]:
REPO_URL = "https://github.com/adikothuri3/PanAF-Ape-Detection.git"
REPO_DIR = "/content/PanAF-Ape-Detection"

import os
from pathlib import Path

if not Path(REPO_DIR).exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
os.environ["PANAF_REPO_ROOT"] = REPO_DIR
print("working directory:", Path.cwd())

## 3. Install

From `requirements-colab.txt`, which is exported from `uv.lock` — so this environment matches the
locked one rather than whatever Colab happens to preinstall.

Several minutes. **If Colab asks you to restart the session, do it**, then continue from section 4
(you do not need to re-run this cell).

In [ ]:
!pip install --quiet -r requirements-colab.txt
!pip install --quiet -e . --no-deps

## 4. Verify the environment — including the device

`doctor` reports the stack; `smoke_inference.py` proves it actually works (imports, ByteTrack,
NumPy interop, a video round-trip) without downloading any weights.

In [ ]:
!python -m panaf_ape_detection.cli doctor

In [ ]:
!python scripts/smoke_inference.py

In [ ]:
# The device resolution the run will use. `auto` prefers cuda -> mps -> cpu.
from panaf_ape_detection.config import load_config
from panaf_ape_detection.runtime import available_devices, resolve_device, set_seeds

config = load_config("configs/colab.yaml")
set_seeds(config.project.seed)

print("available:", sorted(d.value for d in available_devices()))
device = resolve_device(config.model.device)
print(f"configured {config.model.device.value!r} -> resolved {device.value!r}")

if device.value != "cuda":
    print("\nWARNING: not on CUDA. Check the runtime type before running the full pass.")

print("\nmodel:    ", config.model.model_name, "/", config.model.variant)
print("threshold:", config.model.confidence_threshold)
print("max clips:", config.data.max_clips, " frame stride:", config.data.frame_stride)

## 5. Get the clips

Downloads a **purposive** sample directly from the Bristol deposit — no Drive upload needed, the
files are plain HTTPS and roughly 1–6 MB each.

Selection is not random. It profiles candidate *annotations* first (no video), then greedily picks
clips covering the failure axes: all nine behaviours, both species, crowded frames, small and large
subjects, and frames containing no ape at all. The reason for each pick lands in the manifest's
`selected_reason` column.

In [ ]:
!python scripts/fetch_panaf500.py --count 10 --pool 150

In [ ]:
# What was selected, and why.
import pandas as pd

manifest = pd.read_csv("data/sample_manifest.csv")
print(f"{len(manifest)} clips\n")
for _, row in manifest.iterrows():
    print(f"{row.clip_id}  [{row.split}]  {row.species}")
    print(f"    {row.selected_reason}")

## 6. Run detection

Per clip: decode every frame → MegaDetector → confidence filter → compare against ground truth →
draw boxes → stitch to MP4 → write metrics and run metadata.

Watch the log for the device line. If it says `PyTorch-Wildlife ignored device='cuda' ... forcing
it`, that is the upstream bug being corrected — the run continues on the GPU.

Outputs land in `artifacts/`. Each clip is skipped if already complete, so re-running after a
dropped session resumes rather than restarting.

In [ ]:
!panaf-phase1 detect --config configs/colab.yaml

## 7. Look at the results

The metrics below are real measurements at the stated confidence and IoU thresholds. A detection
counts as correct when it **localises an annotated ape** — MegaDetector cannot identify species, so
no species claim is made or implied.

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path("artifacts/metrics").glob("*.json")):
    m = json.loads(path.read_text())
    rows.append({
        "clip": m["clip_id"],
        "frames": m["frames_evaluated"],
        "precision": m["overall"]["precision"],
        "recall": m["overall"]["recall"],
        "f1": m["overall"]["f1"],
        "mean_iou": m["mean_iou"],
        "empty_frames": m["empty_frames"],
        "FP_on_empty": m["false_positives_on_empty_frames"],
    })

metrics = pd.DataFrame(rows)
display(metrics)

if not metrics.empty:
    tp = sum(json.loads(p.read_text())["overall"]["true_positives"]
             for p in Path("artifacts/metrics").glob("*.json"))
    fp = sum(json.loads(p.read_text())["overall"]["false_positives"]
             for p in Path("artifacts/metrics").glob("*.json"))
    fn = sum(json.loads(p.read_text())["overall"]["false_negatives"]
             for p in Path("artifacts/metrics").glob("*.json"))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    print(f"\nOVERALL  TP={tp}  FP={fp}  FN={fn}")
    print(f"         precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

### Where does it fail?

Recall broken down by behaviour and by subject size. This is the table that should drive any
fine-tuning decision — a detector that misses `climbing_up` and small subjects needs different work
from one that misses everything equally.

In [ ]:
from collections import defaultdict

behaviour = defaultdict(lambda: [0, 0])
size = defaultdict(lambda: [0, 0])

for path in Path("artifacts/metrics").glob("*.json"):
    m = json.loads(path.read_text())
    for label, counts in m["by_behaviour"].items():
        behaviour[label][0] += counts["true_positives"]
        behaviour[label][1] += counts["true_positives"] + counts["false_negatives"]
    for band, counts in m["by_size"].items():
        size[band][0] += counts["true_positives"]
        size[band][1] += counts["true_positives"] + counts["false_negatives"]

print("Recall by behaviour")
for label, (found, total) in sorted(behaviour.items(), key=lambda kv: kv[1][0] / max(kv[1][1], 1)):
    print(f"  {label:20} {found:5}/{total:<5} {found / total:.3f}" if total else f"  {label}: n/a")

print("\nRecall by subject size (fraction of frame area)")
for band in ("small", "medium", "large"):
    if band in size:
        found, total = size[band]
        print(f"  {band:8} {found:5}/{total:<5} {found / total:.3f}")

## 8. Watch an annotated clip

**Green = MegaDetector prediction. Amber = dataset ground truth**, labelled with the behaviour and
the individual's id. The legend is burned into every frame so a still pulled out of the video is
still unambiguous.

In [ ]:
import base64
from pathlib import Path

from IPython.display import HTML, display

videos = sorted(Path("artifacts/videos").glob("*_annotated.mp4"))
print(f"{len(videos)} annotated clips")

# Colab cannot play mp4v directly, so re-encode one to H.264 for the player.
if videos:
    source = videos[0]
    playable = source.with_name(source.stem + "_h264.mp4")
    !ffmpeg -y -loglevel error -i "{source}" -vcodec libx264 -pix_fmt yuv420p "{playable}"
    encoded = base64.b64encode(playable.read_bytes()).decode()
    display(HTML(f'<video width=720 controls><source src="data:video/mp4;base64,{encoded}" '
                 f'type="video/mp4"></video>'))
    print(source.name)

## 9. Save the outputs

Colab sessions are ephemeral — **anything not copied out is lost**.

`artifacts/` is git-ignored, and the clips and annotated video must not be committed: they are
derived works of a non-commercially-licensed dataset.

In [ ]:
USE_DRIVE = False   # set True to copy artifacts to Drive
DESTINATION = "/content/drive/MyDrive/panaf-ape-detection/artifacts"

if USE_DRIVE:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    shutil.copytree("artifacts", DESTINATION, dirs_exist_ok=True)
    print("copied to", DESTINATION)
else:
    print("Drive copy disabled. Set USE_DRIVE = True to keep the outputs.")

In [ ]:
# The run-metadata record: commit, config, device, variant, threshold, seed,
# input checksums and elapsed time. This is what makes the run reproducible.
import json
from pathlib import Path

for path in sorted(Path("artifacts/metadata").glob("*.json"))[-1:]:
    meta = json.loads(path.read_text())
    for key in ("experiment_name", "git_commit", "git_dirty", "device",
                "model_variant", "confidence_threshold", "seed", "elapsed_seconds"):
        print(f"{key:22} {meta.get(key)}")
    print(f"{'inputs':22} {len(meta.get('inputs', []))} files, checksummed")

## Next

- Record what you saw in `experiments/experiment_log.md`, **including anything that failed**.
- Fill in `reports/phase1_writeup_template.md` from these numbers — copy it to a dated file first.
- Tracking (ByteTrack) is the next pass. The dataset's `ape_id` gives ground-truth tracks to check
  against, and `TrackedFrameDetections` is already in the schema waiting for it.

Nothing here says anything about species. If the recall table shows a specific weakness — small
subjects, or a particular behaviour — that is the evidence a fine-tuning decision should rest on.